In [ ]:
import os
import pandas as pd
from langchain_openai import ChatOpenAI

need_translation = False
if need_translation:
    df = pd.read_csv('lausanne_map_pack.csv')
    key_abs_path = '/Users/viaccoz/Documents/GitHub/europeana/api_keys/openapikey.txt'
    with open(key_abs_path, 'r') as f:
        cont = f.readlines()
    params = dict((v.replace('\n', '').split('=')) for v in cont)
    # TODO: make it so the output does not have "Here is the translation" at the beginning or similar contextual phrases.
    translation_prompt = """
    You are a professional translator specialized in historical and cartographic terminology. Translate the following {lang1} text into {lang2}, ensuring that all technical terms related to maps and history are accurately conveyed. Maintain the original meaning and context while adapting the language for clarity and precision in English.
    """

    client = ChatOpenAI(
        openai_organization=params['organization_id'],
        openai_api_key=params['api_key'],
        model='gpt-4.1',
        temperature=0.7
    )

    def translate_text(text, source_lang, target_lang):
        prompt = translation_prompt.format(lang1=source_lang, lang2=target_lang)
        response = client.predict(f"{prompt}\n\nText to translate:\n{text}")
        return response.strip()


    df['description_en'] = df.apply(lambda row: translate_text(row['description'], 'French', 'English'), axis=1)
    df['description_it'] = df.apply(lambda row: translate_text(row['description'], 'French', 'Italian'), axis=1)
    df['description_de'] = df.apply(lambda row: translate_text(row['description'], 'French', 'German'), axis=1)
    df['nom_fr'] = df.apply(lambda row: translate_text(row['nom'], 'English', 'French'), axis=1)
    df['nom_it'] = df.apply(lambda row: translate_text(row['nom'], 'English', 'Italian'), axis=1)
    df['nom_de'] = df.apply(lambda row: translate_text(row['nom'], 'English', 'German'), axis=1)
    df['paradata_process_fr'] = df.apply(lambda row: translate_text(row['paradata_process'], 'English', 'French'), axis=1)
    df['paradata_process_it'] = df.apply(lambda row: translate_text(row['paradata_process'], 'English', 'Italian'), axis=1)
    df['paradata_process_de'] = df.apply(lambda row: translate_text(row['paradata_process'], 'English', 'German'), axis=1)
    df['paradata_provenance_en'] = df.apply(lambda row: translate_text(row['paradata_provenance'], 'French', 'English'), axis=1)
    df['paradata_provenance_it'] = df.apply(lambda row: translate_text(row['paradata_provenance'], 'French', 'Italian'), axis=1)
    df['paradata_provenance_de'] = df.apply(lambda row: translate_text(row['paradata_provenance'], 'French', 'German'), axis=1)

    df.to_csv('lausanne_map_pack_translated.csv', index=False)
else:
    df = pd.read_csv('lausanne_map_pack_translated.csv')


In [ ]:

map_script_template = '''
import sys
import os
# to retrieve the utils function used by all notebooks
parent_dir = os.path.abspath('../../../')
if parent_dir not in sys.path: sys.path.insert(0, parent_dir)
from timeatlas.data_modeling import *
from utils.rde import RDE

BASE_SLUG = '{slug}'
MAP_SLUG = f"{{BASE_SLUG}}-map"
BEGIN_TR = {begin_tr}
END_TR = {end_tr}
tuple_TR = (datetime_obj_from_int_time(BEGIN_TR), datetime_obj_from_int_time(END_TR, match_to_end=True))
MAP_UUID = str(uuid.uuid5(VMAP_UUID5_NS, MAP_SLUG))
bm_slug = f"{{MAP_SLUG}}-base"
basemap_layer_uuid = str(uuid.uuid5(VMAP_UUID5_NS, bm_slug))
zoom_lvl= [11,21]

lausanne_area_uuids = get_area_uuids_from_slugs('../../areas/data', ['city-lausanne-area', 'country-switzerland-area'])

bm_layer = produce_layer_obj(basemap_layer_uuid,
                                        bm_slug,
                                         {{ 
                                         "en": ["Historical Map of Lausanne from {start_time}"],
                                         "fr": ["Carte historique de Lausanne de {start_time}"],
                                         "it": ["Mappa storica di Losanna dal {start_time}"],
                                         "de": ["Historische Karte von Lausanne aus dem Jahr {start_time}"]
                                         }},
                                         {{
                                         "en": ["{paradata_process}"],
                                         "fr": ["{paradata_process_fr}"],
                                         "it": ["{paradata_process_it}"],
                                         "de": ["{paradata_process_de}"]
                                         }},
                                         tuple_TR,
                                         MAP_UUID,
                                         is_vector=False,
                                         layer_configs=[produce_layer_config(
                                          str(uuid.uuid5(VMAP_UUID5_NS, f'{{bm_slug}}-config-1')),
                                          zoom_lvl=zoom_lvl,
                                          access_url="https://geo-timemachine.epfl.ch/geoserver/gwc/service/wmts/rest/TimeMachine:{slug}/raster/EPSG:900913x2/EPSG:900913x2:{{z}}/{{y}}/{{x}}?format=image/png",
                                          format='wmts'
                                          )]
                                        )

layers = [bm_layer]
layer_ids = [l['uuid'] for l in layers]
map_obj = produce_map_obj(MAP_UUID, 
                             MAP_SLUG,
                             {{
                              "fr": ["{nom_fr}"],
                              "it": ["{nom_it}"],
                              "de": ["{nom_de}"],
                              "en": ["{nom}"]
                             }},
                             {{
                             "fr": ["{description}"],
                             "it": ["{description_it}"],
                             "de": ["{description_de}"],
                             "en": ["{description_en}"]
                             }},
                             {{
                             "en": ["{paradata_process}"],
                             "fr": ["{paradata_process_fr}"],
                             "it": ["{paradata_process_it}"],
                             "de": ["{paradata_process_de}"]
                             }},
                              "https://image-timemachine.epfl.ch/iiif/3/venice%2Flayer_thumbnails%2F{slug}.png/full/max/0/default.jpg",
                              "1.0",
                              tuple_TR,
                              layer_ids,
                              areas_id=lausanne_area_uuids
                             )

save_data_file_if_different('', 'map', [map_obj], '{slug}_map', RDEType.MAP.value)
save_data_file_if_different('', 'layers', layers, '{slug}_layers', RDEType.LAYER.value)

'''
for index, row in df.iterrows():
    being_tr = int(row['start_time'])*10000 + 101
    end_tr = int(row['end_time'])*10000 + 1231
    script_content = map_script_template.format(
        slug=row['slug'],
        start_time=row['start_time'],
        begin_tr=being_tr,
        end_tr=end_tr,
        nom=row['nom'],
        description=row['description'],
        nom_fr=row['nom_fr'],
        nom_it=row['nom_it'],
        nom_de=row['nom_de'],
        description_en=row['description_en'],
        description_it=row['description_it'],
        description_de=row['description_de'],
        paradata_process_fr=row['paradata_process_fr'],
        paradata_process_it=row['paradata_process_it'],
        paradata_process_de=row['paradata_process_de'],
        paradata_process=row['paradata_process']
    )
    folder_name = row['slug']
    if not os.path.exists(folder_name):
        os.makedirs(folder_name)
    script_filename = f"produce_{row['slug'].replace('-', '_')}_map.py"
    with open(os.path.join(folder_name, script_filename), 'w') as script_file:
        script_file.write(script_content)

# compact version (in one file for everything)

In [ ]:
import sys
import os
# to retrieve the utils function used by all notebooks
parent_dir = os.path.abspath('../../')
if parent_dir not in sys.path: sys.path.insert(0, parent_dir)
from timeatlas.data_modeling import *
from utils.rde import RDE
from ast import literal_eval

df = pd.read_csv('lausanne_map_pack_with_extents.csv')

print(len(df))
df = df[df['slug'] != 'lausanne-1838-berney-centre']
print(len(df))

layers = []
maps = []
for index, r in df.iterrows():
    slug = r['slug']
    extent = literal_eval(r['extent'])
    begin_tr = int(r['start_time'])*10000 + 101
    end_tr = int(r['end_time'])*10000 + 1231
    BASE_SLUG = slug
    MAP_SLUG = f"{BASE_SLUG}-map"
    BEGIN_TR = begin_tr
    END_TR = end_tr
    tuple_TR = (datetime_obj_from_int_time(BEGIN_TR), datetime_obj_from_int_time(END_TR, match_to_end=True))
    MAP_UUID = str(uuid.uuid5(VMAP_UUID5_NS, MAP_SLUG))
    bm_slug = f"{MAP_SLUG}-base"
    basemap_layer_uuid = str(uuid.uuid5(VMAP_UUID5_NS, bm_slug))
    zoom_lvl= [11,21]

    lausanne_area_uuids = get_area_uuids_from_slugs('../areas/data', ['city-lausanne-area', 'country-switzerland-area'])

    bm_layer = produce_layer_obj(basemap_layer_uuid,
                                            bm_slug,
                                            {
                                            "en": [f"Historical Map of Lausanne from {r['start_time']}"],
                                            "fr": [f"Carte historique de Lausanne de {r['start_time']}"],
                                            "it": [f"Mappa storica di Losanna dal {r['start_time']}"],
                                            "de": [f"Historische Karte von Lausanne aus dem Jahr {r['start_time']}"]
                                            },
                                            {
                                            "en": [r['paradata_process']],
                                            "fr": [r['paradata_process_fr']],
                                            "it": [r['paradata_process_fr']],
                                            "de": [r['paradata_process_de']]
                                            },
                                            tuple_TR,
                                            MAP_UUID,
                                            is_vector=False,
                                            layer_configs=[produce_layer_config(
                                            str(uuid.uuid5(VMAP_UUID5_NS, f'{bm_slug}-config-1')),
                                            zoom_lvl=zoom_lvl,
                                            extent=extent,
                                            access_url=f"https://geo-timemachine.epfl.ch/geoserver/gwc/service/wmts/rest/TimeMachine:{slug}/raster/EPSG:900913x2/EPSG:900913x2:{{z}}/{{y}}/{{x}}?format=image/png",
                                            format='wmts'
                                            )]
                                            )

    layers.append(bm_layer)
    map_obj = produce_map_obj(MAP_UUID, 
                                MAP_SLUG,
                                {
                                "fr": [r['nom_fr']],
                                "it": [r['nom_it']],
                                "de": [r['nom_de']],
                                "en": [r['nom']]
                                },
                                {
                                "fr": [r['description']],
                                "it": [r['description_it']],
                                "de": [r['description_de']],
                                "en": [r['description_en']]
                                },
                                {
                                "en": [r['paradata_process']],
                                "fr": [r['paradata_process_fr']],
                                "it": [r['paradata_process_it']],
                                "de": [r['paradata_process_de']]
                                }, 
                                f"https://image-timemachine.epfl.ch/iiif/3/lausanne%2Flayer_thumbnails%2F{slug}.png/full/max/0/default.jpg",
                                "1.0",
                                tuple_TR,
                                layers,
                                areas_id=lausanne_area_uuids
                                )
    maps.append(map_obj)

folder_save_fp = f'lausanne-{min(df["start_time"])}-to-{max(df["end_time"])}-maps'
if not os.path.exists(folder_save_fp):
    os.makedirs(folder_save_fp)
save_data_file_if_different(folder_save_fp, 'map', maps, '{slug}_map', RDEType.MAP.value)
save_data_file_if_different(folder_save_fp, 'layers', layers, '{slug}_layers', RDEType.LAYER.value)

In [ ]:

def generate_thumbnail_from_raster_url_and_extent_box(raster_url, extent_box, save_fp, thumb_size=(256,256)):
    import rasterio
    from rasterio.plot import reshape_as_image
    from PIL import Image
    import numpy as np
    from io import BytesIO
    import requests

    response = requests.get(raster_url)
    raster_data = BytesIO(response.content)

    with rasterio.open(raster_data) as src:
        window = src.window(*extent_box)
        transform = src.window_transform(window)
        data = src.read(window=window)
        img_array = reshape_as_image(data)

        img = Image.fromarray(img_array)
        img.thumbnail(thumb_size)
        img.save(save_fp)
        
raster_url_fmt = "https://geo-timemachine.epfl.ch/geoserver/gwc/service/wmts/rest/TimeMachine:{slug}/raster/EPSG:900913x2/EPSG:900913x2:{{z}}/{{y}}/{{x}}?format=image/png"
slug_t = 'lausanne-1723-melotte-owners'
raster_url = raster_url_fmt.format(slug=slug_t)
generate_thumbnail_from_raster_url_and_extent_box(
    raster_url,
    extent_box=(255000, 124000, 257000, 126000),
    save_fp='thumbnail_test.png'
)

In [ ]:
import rasterio
from rasterio.plot import show
from rasterio.windows import from_bounds
from pyproj import Transformer
import os
import argparse
import matplotlib.pyplot as plt
import owslib.wms
from owslib.wmts import WebMapTileService

extent_dict = {
    'rialto': [45.439886196, 12.339491847, 45.436191552, 12.331912182],
    'lausanne_cathedral': [46.527416320, 6.642246776, 46.517441929, 6.627888480]
}


raster_url_fmt = "https://geo-timemachine.epfl.ch/geoserver/gwc/service/wmts/rest/TimeMachine:{slug}/raster/EPSG:900913x2"#/EPSG:900913x2:{{z}}/{{y}}/{{x}}?format=image/png"
slug_t = 'lausanne-1723-melotte-owners'
raster_url = raster_url_fmt.format(slug=slug_t)

def generate_thumbnail_from_wms_url(wms_url:str, output_path:str, extent_nesw:list[float], width=1024, height=768):
    
    wms = owslib.wms.WebMapService(wms_url, version='1.3.0')
    layer_name = list(wms.contents)[0]
    img = wms.getmap(
        layers=[layer_name],
        srs='EPSG:4326',
        bbox=(extent_nesw[3], extent_nesw[2], extent_nesw[1], extent_nesw[0]),
        size=(width, height),
        format='image/png',
        transparent=True
    )
    with open(output_path, 'wb') as out:
        out.write(img.read())

def generate_thumbnail_from_wmts_url(wmts_service_url:str,  slug:str, output_path:str, extent_nesw:list[float], width=1024, height=768):
    wmts = WebMapTileService(wmts_service_url)
    found_layer = None
    for layer_name, _ in wmts.contents.items():
        if slug in layer_name:
            found_layer = layer_name
            break
    if not found_layer:
        raise ValueError(f"Layer with slug {slug} not found in WMTS service.")
    layer_name = found_layer
    
    # Get layer details
    layer = wmts[layer_name]
    tilematrixset = list(layer.tilematrixsets)[0]  # Use first available tilematrixset
    
    # Get an appropriate tilematrix (zoom level) - using a middle zoom level
    available_matrices = list(layer.tilematrixsetlinks[tilematrixset].tilematrixlimits.keys())
    tilematrix = available_matrices[len(available_matrices)//2] if available_matrices else '0'
    print(tilematrix)
    # Get tile at center position (simplified - just getting one tile as thumbnail)
    tile = wmts.gettile(
        layer=layer_name,
        tilematrixset=tilematrixset,
        tilematrix=tilematrix,
        row=494,
        column=2123,
        format="image/png"
    )
    
    # Write tile to output file
    with open(output_path, 'wb') as out:
        out.write(tile.read())

# generate_thumbnail_from_wmts_url(raster_url, 'thumbnail_from_wms.png', extent_dict['lausanne_cathedral'])

raster_url_fmt = "https://geo-timemachine.epfl.ch/geoserver/gwc/service/wmts/rest/TimeMachine:{slug}/raster/EPSG:900913x2/EPSG:900913x2:{{z}}/{{y}}/{{x}}?format=image/png"
# t = 'https://geo-timemachine.epfl.ch/geoserver/gwc/service/wmts/rest/TimeMachine:lausanne-1824-spengler/raster/EPSG:900913x2/EPSG:900913x2:16/23179/33972?format=image/png'
# t = 'http://map1c.vis.earthdata.nasa.gov/wmts-geo/wmts.cgi'
# t = 'https://geo-timemachine.epfl.ch/geoserver/gwc/service/wmts'
# raster_url = raster_url_fmt.format(slug=slug_t)
# print(raster_url)
# wmts = WebMapTileService(t)
service_url = 'https://geo-timemachine.epfl.ch/geoserver/gwc/service/wmts'
generate_thumbnail_from_wmts_url(service_url, slug_t, 'thumbnail_from_wms.png', extent_dict['lausanne_cathedral'])


In [ ]:
import rasterio
from rasterio.plot import show
from rasterio.windows import from_bounds
from pyproj import Transformer
import os
import argparse
import matplotlib.pyplot as plt
import owslib.wms
from owslib.wmts import WebMapTileService
from shapely.geometry import Point


def get_extent_from_wmts(wmts_service_url: str, slug: str) -> list[float]:
    """
    Derives the geographic extent from a WMTS service for a given layer.
    
    Args:
        wmts_service_url: The base URL of the WMTS service
        slug: The slug identifier to find the specific layer
        
    Returns:
        List of 4 GPS coordinates in format [north, east, south, west] in EPSG:4326
    """
    from owslib.wmts import WebMapTileService
    from pyproj import Transformer
    
    wmts = WebMapTileService(wmts_service_url)
    
    # Find the layer by slug
    found_layer = None
    for layer_name in wmts.contents.keys():
        # print(wmts.cont)
        if slug in layer_name:
            found_layer = layer_name
            break
    if not found_layer:
        raise ValueError(f"Layer with slug '{slug}' not found in WMTS service.")
    
    layer = wmts[found_layer]
    
    # Get bounding box - usually in the native CRS
    bbox = layer.boundingBoxWGS84
    
    if bbox:
        # bbox format is typically (minx, miny, maxx, maxy) or (west, south, east, north)
        west, south, east, north = bbox
        return [north, east, south, west]
    
    # If WGS84 bbox not available, try to get from tilematrixset and transform
    tilematrixset = list(layer.tilematrixsets)[0]
    bbox_native = layer.boundingBox
    
    if bbox_native:
        minx, miny, maxx, maxy = bbox_native[:4]
        native_crs = bbox_native[4] if len(bbox_native) > 4 else "EPSG:3857"
        
        # Transform to WGS84 if needed
        if native_crs != "EPSG:4326":
            transformer = Transformer.from_crs(native_crs, "EPSG:4326", always_xy=True)
            west, south = transformer.transform(minx, miny)
            east, north = transformer.transform(maxx, maxy)
        else:
            west, south, east, north = minx, miny, maxx, maxy
        
        return [north, east, south, west]
    
    raise ValueError(f"No bounding box information available for layer '{found_layer}'")

import pandas as pd
df = pd.read_csv('lausanne_map_pack_translated.csv')

service_url = 'https://geo-timemachine.epfl.ch/geoserver/gwc/service/wmts'
extent_d = {}
for idx, row in df.iterrows():
    try:
        slug = row['slug']
        if 'lausanne-1638-maquette-mhl' in slug:
            extent_d[slug] = ['POINT (6.641402451544036 46.526546235994196)', 'POINT (6.626308575527873 46.51849249768184)']
            continue
        # Test the function
        extent = get_extent_from_wmts(service_url, slug)
        ne_point = Point(extent[1], extent[0]).wkt
        sw_point = Point(extent[3], extent[2]).wkt
        extent_d[slug] = [ne_point, sw_point]
        # print(f"Extent for {slug_test}: {extent}")
        # print(f"Format: [north={extent[0]}, east={extent[1]}, south={extent[2]}, west={extent[3]}]")
    except ValueError as e:
        print(f"Layer not found or no bounding box available: {e}")

df['extent'] = df['slug'].apply(lambda slug: extent_d.get(slug, None))
df.to_csv('lausanne_map_pack_with_extents.csv', index=False)

In [ ]:
non_lausanne_pack_slugs = [ 
 '1831_Berney',
 '1831_Berney_LZW',
 'lausanne_cadastre_berney_v7_7',
 'geometries_VENICE_1808_SOMMARIONI_CADASTER_MAP',
 'venice-1729-lughi',
 'venice-1846-begcombatti2',
 'venice-1846-nbleventhal',
 'garzoni_geometries',
 'venice-1834-bggallo',
 'obs_VENICE_GARZONI',
 'venice-1729-hlibrary',
 'venice-1838-wbclarke',
 '1832_almanach',
 'venice-1846-begcombatti',
 'sommarioni_geometries_rde_20240522',
 'geometries_VENICE_1740_PARISH_MAP'
]
extend_d = {}
for v in non_lausanne_pack_slugs:
    try:
         slug = v
         # Test the function
         extent = get_extent_from_wmts(service_url, slug)
         nw_point = Point(extent[3], extent[0]).wkt
         se_point = Point(extent[1], extent[2]).wkt
         extend_d[slug] = [nw_point, se_point]
    except ValueError as e:
         print(f"Layer not found or no bounding box available: {e}")

In [ ]:

import json
from shapely.geometry import shape, Polygon, LineString, Point
from shapely.ops import unary_union
import json


def geometry_parser(geometry):
    if geometry['type'] == 'Polygon':
        return Polygon(geometry['coordinates'][0])
    elif geometry['type'] == 'MultiPolygon':
        polygons = [Polygon(p[0]) for p in geometry['coordinates']]
        return unary_union(polygons)
    elif geometry['type'] == 'Point':
        return Point(geometry['coordinates'])
    elif geometry['type'] == 'LineString':
        return LineString(geometry['coordinates'])
    elif geometry['type'] == 'MultiLineString':
        lines = [LineString(line) for line in geometry['coordinates']]
        return unary_union(lines)
    elif geometry['type'] == 'GeometryCollection':
        shapes = [geometry_parser(geom) for geom in geometry['geometries']]
        return unary_union(shapes)
    else:
        raise ValueError(f"Unsupported geometry type: {geometry['type']}")

def generate_extent_from_geometry_list(geometry_list):

    shapes = [geometry_parser(geom['geometry']) for geom in geometry_list]
    combined_shape = unary_union(shapes)
    minx, miny, maxx, maxy = combined_shape.bounds
    nw_point = Point(minx, maxy).wkt
    se_point = Point(maxx, miny).wkt
    return [nw_point, se_point]

geometry_fp = '../datasets/lausanne-qbuildings/src/qbuildings-lausanne.geojson'
# with open(geometry_fp, 'r') as f:
#     data = json.load(f)['rde_objects']
# generate_extent_from_geometry_list(data)

# for native geometries source
import geopandas as gpd
gdf = gpd.read_file('../datasets/lausanne-qbuildings/src/qbuildings-lausanne.geojson')
combined_shape = unary_union(gdf['geometry'])
minx, miny, maxx, maxy = combined_shape.bounds
nw_point = Point(minx, maxy).wkt
se_point = Point(maxx, miny).wkt
print([nw_point, se_point])

In [ ]:
import rasterio
from rasterio.plot import show
from rasterio.windows import from_bounds
from pyproj import Transformer
import os
import argparse
import matplotlib.pyplot as plt
import owslib.wms


extent_dict = {
    'rialto': [45.439886196, 12.339491847, 45.436191552, 12.331912182],
    'lausanne_cathedral': [46.527416320, 6.642246776, 46.517441929, 6.627888480]
}

def generate_thumbnail_from_geotiff(geotiff_path:str, output_path:str, extent_nesw:list[float], width=1024, height=768):
    with rasterio.open(geotiff_path) as src:
        geotiff_crs = src.crs
        
        # Define the transformer to convert from WGS:84 to the GeoTIFF's CRS
        transformer = Transformer.from_crs("EPSG:4326", geotiff_crs, always_xy=True)
        
        # Reproject the extent to the GeoTIFF's CRS & create a window from the reprojected extent
        west, south = transformer.transform(extent_nesw[3], extent_nesw[2])
        east, north = transformer.transform(extent_nesw[1], extent_nesw[0])
        window = from_bounds(west, south, east, north, src.transform)
        data = src.read(window=window)
        _, ax = plt.subplots(figsize=(width/100, height/100), dpi=100)
        show(data, transform=src.window_transform(window), ax=ax)
        
        # Save the plot as a PNG
        plt.axis('off')
        plt.savefig(output_path, bbox_inches='tight', pad_inches=0)
        plt.close()


def resize_image(input_path:str, output_path:str, size=(256,256)):
    from PIL import Image
    img = Image.open(input_path)
    img.thumbnail(size)
    img.save(output_path)

FOLDER_SRC = '../../utils/drive-download-20240917T153759Z-001'
for img in reversed(os.listdir(FOLDER_SRC)):
    # if 'lsne' in img.lower() and img.endswith('.tif'):
    #     print(f"Generating thumbnail for {img}...")
    #     lsne_code = img.replace('plan_lsne_', '').split('-')[0]
    #     year_part = lsne_code[:4]
    #     code_part = lsne_code[-3:]
    #     lsne_stub = f"{year_part}_plan_lsne_{code_part}"
    #     print(lsne_stub)
    #     slug = df[df['georeferenced map filename'].str.contains(lsne_stub)]['slug'].values[0]
    #     generate_thumbnail_from_geotiff(
    #         os.path.join(FOLDER_SRC, img),
    #         slug,
    #         extent_dict['rialto']
    #     )
        
    if '.tif' in img and img in df['document filename'].values:
        print(f"Generating thumbnail for {img}...")
        map_slug = df[df['document filename'] == img]['slug'].values[0]
        generate_thumbnail_from_geotiff(
            os.path.join(FOLDER_SRC, img),
            f'{map_slug}.png',
            extent_dict['lausanne_cathedral']
        )
    elif '.tif' in img:
        print(f"Skipping {img}, not in the map pack.")
